In [2]:
# ============================================================
# MULTI-TEACHER ENSEMBLE EVALUATION
# (MultiScaleSleepNetPlain + Context-CNN-Transformer-SupCon
#  + TCN-BiLSTM)
#
# NO RETRAINING
#
# Level 1:
#   5 seeds -> average softmax probability
#
# Level 2:
#   T1 + T2 + T3 -> equal 1/3 soft-vote
#
# SAVES:
#   ensemble_results.csv
#   ensemble_results.txt
#   val_combined_preds.npy
#   test_combined_preds.npy
#   val_labels.npy
#   test_labels.npy
# ============================================================

import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score
)


# ============================================================
# PATHS & CONFIG
# ============================================================

DATA_PATH = r"D:\22\AA\preprocess\preprocessed_FFinal"

SPLIT_PATH = r"D:\22\AA\AA journal\preprocess\preprocessed_split_v2"

T1_CKPT_DIR = (
    r"D:\22\AA\AA journal\evaluation"
    r"\multiscale_plain_c7_valfixed"
)

T2_CKPT_DIR = (
    r"D:\22\AA\AA journal\evaluation"
    r"\context_cnn_transformer_c7_supcon_valfixed"
)

T3_CKPT_DIR = (
    r"D:\22\AA\AA journal\evaluation"
    r"\tcn_bilstm_c7_valfixed"
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = (
    r"D:\22\AA\AA journal\evaluation"
    r"\multi_teacher_ensemble"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# GENERAL CONFIG
# ============================================================

LABEL_NAMES = [
    "Wake",
    "N1",
    "N2",
    "N3",
    "REM"
]

CONTEXT = 7

WINDOW = 2 * CONTEXT + 1

# WINDOW = 15

BATCH_SIZE = 64

SEEDS = [
    42,
    123,
    256,
    789,
    999
]


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

print(
    "Multi-teacher ensemble:"
    " T1 (MultiScaleSleepNetPlain)"
    " + T2 (Context-CNN-Transformer-SupCon)"
    " + T3 (TCN-BiLSTM)"
)

print(f"Output directory: {OUTPUT_DIR}")


# ============================================================
# 3-WAY SPLIT
# ============================================================

_val_path = os.path.join(
    SPLIT_PATH,
    "_val_subs.npy"
)

_test_path = os.path.join(
    SPLIT_PATH,
    "_test_subs.npy"
)


for p, name in [
    (_val_path, "_val_subs.npy"),
    (_test_path, "_test_subs.npy")
]:

    if not os.path.exists(p):

        raise FileNotFoundError(
            f"{name} not found at {SPLIT_PATH}"
        )


VAL_SUBS = np.load(
    _val_path,
    allow_pickle=True
).tolist()


TEST_SUBS = np.load(
    _test_path,
    allow_pickle=True
).tolist()


print(
    f"Val subjects: {len(VAL_SUBS)}"
    f"   Test subjects: {len(TEST_SUBS)}"
)


# ============================================================
# DATASET
#
# PSG channels:
#   F3
#   C3
#   EOG1
#
# Context = 7
# Window = 15
# ============================================================

class PSGContextDataset(Dataset):

    def __init__(
        self,
        subject_list,
        data_path,
        context=CONTEXT
    ):

        self.context = context

        self.data = []

        self.index = []


        for sub in subject_list:

            fp = os.path.join(
                data_path,
                f"{sub}.npz"
            )


            if not os.path.exists(fp):

                print(
                    f"WARNING: missing {fp}"
                )

                continue


            with np.load(fp) as d:

                eeg = d["eeg"][:, [0, 1], :]

                eog = d["eog"][:, [0], :]

                signal = np.concatenate(
                    [eeg, eog],
                    axis=1
                ).astype(np.float32)

                labels = d["labels"].copy()


            n = len(labels)

            sub_idx = len(self.data)


            self.data.append(
                (
                    signal,
                    labels
                )
            )


            for i in range(n):

                self.index.append(
                    (
                        sub_idx,
                        i,
                        n
                    )
                )


        total = len(self.index)


        print(
            f"  Subjects: {len(self.data)}"
            f"   Samples: {total:,}"
        )


    def __len__(self):

        return len(self.index)


    def __getitem__(self, idx):

        sub_idx, center_i, n = self.index[idx]


        signal, labels = self.data[sub_idx]


        window_epochs = []


        for offset in range(
            -self.context,
            self.context + 1
        ):

            ei = max(
                0,
                min(
                    n - 1,
                    center_i + offset
                )
            )


            window_epochs.append(
                signal[ei]
            )


        x = np.stack(
            window_epochs,
            axis=0
        )


        y = int(
            labels[center_i]
        )


        return (
            torch.FloatTensor(x),
            torch.tensor(
                y,
                dtype=torch.long
            )
        )


# ============================================================
# BUILD DATASETS
# ============================================================

print("\nBuilding VAL dataset:")

val_ds = PSGContextDataset(
    VAL_SUBS,
    DATA_PATH
)


print("Building TEST dataset:")

test_ds = PSGContextDataset(
    TEST_SUBS,
    DATA_PATH
)


if len(val_ds) == 0 or len(test_ds) == 0:

    raise RuntimeError(
        "Empty dataset -- check DATA_PATH."
    )


val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# ============================================================
# ============================================================
# T1: MultiScaleSleepNetPlain
# ============================================================
# ============================================================

T1_D_MODEL = 128

T1_DROPOUT = 0.4

T1_N_HEADS = 4


class SEBlock(nn.Module):

    def __init__(
        self,
        channels,
        reduction=8
    ):

        super().__init__()


        self.pool = nn.AdaptiveAvgPool1d(1)


        self.fc = nn.Sequential(

            nn.Linear(
                channels,
                channels // reduction
            ),

            nn.ReLU(),

            nn.Linear(
                channels // reduction,
                channels
            ),

            nn.Sigmoid()
        )


    def forward(self, x):

        b, c, _ = x.shape


        s = self.pool(x).view(
            b,
            c
        )


        s = self.fc(s).view(
            b,
            c,
            1
        )


        return x * s


class MultiScaleCNN(nn.Module):

    def __init__(
        self,
        in_ch=3,
        d_model=T1_D_MODEL,
        dropout=T1_DROPOUT
    ):

        super().__init__()


        mid = d_model // 4


        def time_branch(kernel):

            return nn.Sequential(

                nn.Conv1d(
                    in_ch,
                    mid,
                    kernel_size=kernel,
                    stride=6,
                    padding=kernel // 2
                ),

                nn.BatchNorm1d(mid),

                nn.GELU(),

                nn.MaxPool1d(
                    4,
                    4
                ),

                nn.Conv1d(
                    mid,
                    mid,
                    kernel_size=8,
                    padding=4
                ),

                nn.BatchNorm1d(mid),

                nn.GELU(),

                nn.MaxPool1d(
                    2,
                    2
                ),

                nn.Dropout(dropout)
            )


        self.small = time_branch(25)

        self.medium = time_branch(50)

        self.large = time_branch(100)


        self.spectral = nn.Sequential(

            nn.Conv1d(
                in_ch,
                mid,
                kernel_size=25,
                stride=6,
                padding=12
            ),

            nn.BatchNorm1d(mid),

            nn.GELU(),

            nn.MaxPool1d(
                4,
                4
            ),

            nn.Conv1d(
                mid,
                mid,
                kernel_size=8,
                padding=4
            ),

            nn.BatchNorm1d(mid),

            nn.GELU(),

            nn.MaxPool1d(
                2,
                2
            ),

            nn.Dropout(dropout)
        )


        with torch.no_grad():

            dummy = torch.zeros(
                1,
                in_ch,
                3000
            )


            L_s = self.small(
                dummy
            ).shape[2]


            L_m = self.medium(
                dummy
            ).shape[2]


            L_l = self.large(
                dummy
            ).shape[2]


            L_f = self.spectral(
                dummy
            ).shape[2]


        target_L = min(
            L_s,
            L_m,
            L_l,
            L_f
        )


        self.pool_s = nn.AdaptiveAvgPool1d(
            target_L
        )

        self.pool_m = nn.AdaptiveAvgPool1d(
            target_L
        )

        self.pool_l = nn.AdaptiveAvgPool1d(
            target_L
        )

        self.pool_f = nn.AdaptiveAvgPool1d(
            target_L
        )


        self.out_len = target_L


        self.se = SEBlock(
            4 * mid
        )


        self.proj = nn.Sequential(

            nn.Conv1d(
                4 * mid,
                d_model,
                kernel_size=1
            ),

            nn.BatchNorm1d(
                d_model
            ),

            nn.GELU()
        )


    def forward(self, x):

        fs = self.pool_s(
            self.small(x)
        )


        fm = self.pool_m(
            self.medium(x)
        )


        fl = self.pool_l(
            self.large(x)
        )


        x_fft = torch.fft.rfft(
            x,
            dim=-1
        )


        x_mag = torch.abs(
            x_fft
        )


        if x_mag.shape[-1] < x.shape[-1]:

            pad = (
                x.shape[-1]
                -
                x_mag.shape[-1]
            )


            x_mag = F.pad(
                x_mag,
                (0, pad)
            )


        else:

            x_mag = x_mag[
                ...,
                :x.shape[-1]
            ]


        ff = self.pool_f(
            self.spectral(x_mag)
        )


        feat = torch.cat(
            [
                fs,
                fm,
                fl,
                ff
            ],
            dim=1
        )


        feat = self.se(
            feat
        )


        return self.proj(
            feat
        )


class BiLSTMTransformerBlock(nn.Module):

    def __init__(
        self,
        d_model=T1_D_MODEL,
        n_heads=T1_N_HEADS,
        dropout=T1_DROPOUT
    ):

        super().__init__()


        self.bilstm = nn.LSTM(

            d_model,

            d_model // 2,

            num_layers=1,

            batch_first=True,

            bidirectional=True
        )


        self.norm1 = nn.LayerNorm(
            d_model
        )


        encoder_layer = nn.TransformerEncoderLayer(

            d_model=d_model,

            nhead=n_heads,

            dim_feedforward=d_model * 2,

            dropout=dropout,

            batch_first=True,

            activation="gelu"
        )


        self.transformer = nn.TransformerEncoder(

            encoder_layer,

            num_layers=1
        )


        self.norm2 = nn.LayerNorm(
            d_model
        )


    def forward(self, x):

        lstm_out, _ = self.bilstm(x)


        x = self.norm1(
            x + lstm_out
        )


        attn_out = self.transformer(x)


        return self.norm2(
            x + attn_out
        )


class MultiScaleSleepNetPlain(nn.Module):

    def __init__(
        self,
        in_ch=3,
        d_model=T1_D_MODEL,
        n_layers=2,
        dropout=T1_DROPOUT,
        n_classes=5,
        context=CONTEXT
    ):

        super().__init__()


        self.context = context

        self.d_model = d_model


        self.cnn = MultiScaleCNN(

            in_ch=in_ch,

            d_model=d_model,

            dropout=dropout
        )


        self.intra_blocks = nn.Sequential(
            *[
                BiLSTMTransformerBlock(
                    d_model,
                    dropout=dropout
                )
                for _ in range(n_layers)
            ]
        )


        self.inter_pos = nn.Parameter(

            torch.randn(
                1,
                WINDOW,
                d_model
            ) * 0.01
        )


        self.inter_blocks = nn.Sequential(
            *[
                BiLSTMTransformerBlock(
                    d_model,
                    dropout=dropout
                )
                for _ in range(n_layers)
            ]
        )


        self.classifier = nn.Sequential(

            nn.LayerNorm(d_model),

            nn.Linear(
                d_model,
                64
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                64,
                n_classes
            )
        )


    def forward(self, x):

        B, W, C, T = x.shape


        cnn_out = self.cnn(
            x.view(
                B * W,
                C,
                T
            )
        ).permute(
            0,
            2,
            1
        )


        intra = self.intra_blocks(
            cnn_out
        )


        epoch_feat = intra.mean(
            dim=1
        ).view(
            B,
            W,
            self.d_model
        )


        inter = self.inter_blocks(

            epoch_feat
            +
            self.inter_pos
        )


        center = inter[
            :,
            self.context,
            :
        ]


        return self.classifier(
            center
        )


# ============================================================
# ============================================================
# T3: TCN + BiLSTM
# ============================================================
# ============================================================

T3_TCN_CHANNELS = 128

T3_LSTM_HIDDEN = 64

T3_LSTM_LAYERS = 2

T3_DROPOUT = 0.4


class TCNBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=7,
        dilation=1,
        dropout=0.4
    ):

        super().__init__()


        padding = (
            (kernel_size - 1)
            *
            dilation
        ) // 2


        self.conv1 = nn.Conv1d(

            in_channels,

            out_channels,

            kernel_size,

            padding=padding,

            dilation=dilation
        )


        self.bn1 = nn.BatchNorm1d(
            out_channels
        )


        self.conv2 = nn.Conv1d(

            out_channels,

            out_channels,

            kernel_size,

            padding=padding,

            dilation=dilation
        )


        self.bn2 = nn.BatchNorm1d(
            out_channels
        )


        self.dropout = nn.Dropout(
            dropout
        )


        self.residual = (

            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1
            )

            if in_channels != out_channels

            else nn.Identity()
        )


    def forward(self, x):

        residual = self.residual(x)


        out = self.dropout(

            F.gelu(
                self.bn1(
                    self.conv1(x)
                )
            )
        )


        out = self.dropout(

            F.gelu(
                self.bn2(
                    self.conv2(out)
                )
            )
        )


        return out + residual


class TCNBiLSTM(nn.Module):

    def __init__(
        self,
        in_channels=3,
        tcn_channels=T3_TCN_CHANNELS,
        lstm_hidden=T3_LSTM_HIDDEN,
        lstm_layers=T3_LSTM_LAYERS,
        dropout=T3_DROPOUT,
        n_classes=5,
        context=CONTEXT
    ):

        super().__init__()


        self.context = context

        self.window = (
            2 * context + 1
        )


        self.tcn = nn.Sequential(

            nn.Conv1d(
                in_channels,
                32,
                kernel_size=25,
                stride=4,
                padding=12
            ),

            nn.BatchNorm1d(32),

            nn.GELU(),

            nn.MaxPool1d(
                4,
                4
            ),


            TCNBlock(
                32,
                64,
                kernel_size=7,
                dilation=1,
                dropout=dropout
            ),


            TCNBlock(
                64,
                64,
                kernel_size=7,
                dilation=2,
                dropout=dropout
            ),


            TCNBlock(
                64,
                128,
                kernel_size=7,
                dilation=4,
                dropout=dropout
            ),


            TCNBlock(
                128,
                128,
                kernel_size=7,
                dilation=8,
                dropout=dropout
            ),


            nn.MaxPool1d(
                2,
                2
            ),


            nn.Dropout(dropout)
        )


        with torch.no_grad():

            dummy = torch.zeros(
                1,
                in_channels,
                3000
            )

            self.tcn_seq_len = self.tcn(
                dummy
            ).shape[-1]


        self.intra_pool = nn.AdaptiveAvgPool1d(
            1
        )


        self.feature_norm = nn.LayerNorm(
            tcn_channels
        )


        self.inter_lstm = nn.LSTM(

            input_size=tcn_channels,

            hidden_size=lstm_hidden,

            num_layers=lstm_layers,

            batch_first=True,

            bidirectional=True,

            dropout=(
                dropout
                if lstm_layers > 1
                else 0.0
            )
        )


        self.center_norm = nn.LayerNorm(
            2 * lstm_hidden
        )


        self.classifier = nn.Sequential(

            nn.LayerNorm(
                2 * lstm_hidden
            ),

            nn.Linear(
                2 * lstm_hidden,
                64
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                64,
                n_classes
            )
        )


    def forward(self, x):

        B, W, C, T = x.shape


        feat = self.tcn(
            x.reshape(
                B * W,
                C,
                T
            )
        )


        feat = self.feature_norm(

            self.intra_pool(
                feat
            ).squeeze(-1)
        )


        feat = feat.reshape(
            B,
            W,
            T3_TCN_CHANNELS
        )


        lstm_out, _ = self.inter_lstm(
            feat
        )


        center_feat = self.center_norm(

            lstm_out[
                :,
                self.context,
                :
            ]
        )


        return self.classifier(
            center_feat
        )


# ============================================================
# ============================================================
# T2: Context CNN + Transformer + SupCon
# ============================================================
# ============================================================

T2_D_MODEL = 128

T2_N_HEADS = 8

T2_NUM_LAYERS = 3

T2_DIM_FF = 256

T2_DROPOUT = 0.4

T2_PROJ_DIM = 64


class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=512,
        dropout=0.1
    ):

        super().__init__()


        self.dropout = nn.Dropout(
            dropout
        )


        import math as _math


        pe = torch.zeros(
            max_len,
            d_model
        )


        pos = torch.arange(
            0,
            max_len
        ).unsqueeze(1).float()


        div = torch.exp(

            torch.arange(
                0,
                d_model,
                2
            ).float()

            *
            (
                -_math.log(10000.0)
                /
                d_model
            )
        )


        pe[:, 0::2] = torch.sin(
            pos * div
        )


        pe[:, 1::2] = torch.cos(
            pos * div
        )


        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )


    def forward(self, x):

        return self.dropout(

            x
            +
            self.pe[
                :,
                :x.size(1),
                :
            ]
        )


class ContextCNNTransformerSupCon(nn.Module):

    def __init__(
        self,
        in_ch=3,
        d_model=T2_D_MODEL,
        nhead=T2_N_HEADS,
        num_layers=T2_NUM_LAYERS,
        dim_ff=T2_DIM_FF,
        dropout=T2_DROPOUT,
        n_classes=5,
        context=CONTEXT,
        proj_dim=T2_PROJ_DIM
    ):

        super().__init__()


        self.context = context

        self.d_model = d_model


        self.epoch_cnn = nn.Sequential(

            nn.Conv1d(
                in_ch,
                32,
                kernel_size=50,
                stride=6,
                padding=25
            ),

            nn.BatchNorm1d(32),

            nn.GELU(),

            nn.MaxPool1d(
                4,
                4
            ),


            nn.Conv1d(
                32,
                64,
                kernel_size=8,
                padding=4
            ),

            nn.BatchNorm1d(64),

            nn.GELU(),


            nn.Conv1d(
                64,
                d_model,
                kernel_size=8,
                padding=4
            ),

            nn.BatchNorm1d(
                d_model
            ),

            nn.GELU(),

            nn.MaxPool1d(
                2,
                2
            ),


            nn.Dropout(dropout)
        )


        with torch.no_grad():

            dummy = torch.zeros(
                1,
                in_ch,
                3000
            )


            intra_seq = self.epoch_cnn(
                dummy
            ).shape[2]


        self.intra_pos = PositionalEncoding(

            d_model,

            max_len=intra_seq + 10,

            dropout=dropout
        )


        self.intra_transformer = nn.TransformerEncoder(

            nn.TransformerEncoderLayer(

                d_model=d_model,

                nhead=nhead,

                dim_feedforward=dim_ff,

                dropout=dropout,

                activation="gelu",

                batch_first=True,

                norm_first=True
            ),

            num_layers=1
        )


        self.inter_pos = PositionalEncoding(

            d_model,

            max_len=WINDOW + 2,

            dropout=dropout
        )


        self.inter_transformer = nn.TransformerEncoder(

            nn.TransformerEncoderLayer(

                d_model=d_model,

                nhead=nhead,

                dim_feedforward=dim_ff,

                dropout=dropout,

                activation="gelu",

                batch_first=True,

                norm_first=True
            ),

            num_layers=num_layers
        )


        self.classifier = nn.Sequential(

            nn.LayerNorm(d_model),

            nn.Linear(
                d_model,
                64
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                64,
                n_classes
            )
        )


        self.projector = nn.Sequential(

            nn.LayerNorm(d_model),

            nn.Linear(
                d_model,
                d_model
            ),

            nn.GELU(),

            nn.Linear(
                d_model,
                proj_dim
            )
        )


    def forward(self, x):

        B, W, C, T = x.shape


        feat = self.epoch_cnn(

            x.view(
                B * W,
                C,
                T
            )
        )


        feat = feat.permute(
            0,
            2,
            1
        )


        feat = self.intra_transformer(

            self.intra_pos(
                feat
            )
        )


        feat = feat.mean(
            dim=1
        )


        feat = feat.view(
            B,
            W,
            self.d_model
        )


        feat = self.inter_transformer(

            self.inter_pos(
                feat
            )
        )


        center_feat = feat[
            :,
            self.context,
            :
        ]


        logits = self.classifier(
            center_feat
        )


        proj = F.normalize(

            self.projector(
                center_feat
            ),

            dim=1
        )


        return logits, proj


# ============================================================
# LOAD CHECKPOINTS
# ============================================================

def load_checkpoints(
    model_class,
    ckpt_dir,
    seeds,
    **model_kwargs
):

    models = []


    for seed in seeds:

        ckpt = os.path.join(

            ckpt_dir,

            f"best_seed{seed}.pt"
        )


        if not os.path.exists(ckpt):

            print(
                f"  MISSING: {ckpt}"
            )

            continue


        print(
            f"  Loading seed {seed}..."
        )


        m = model_class(
            **model_kwargs
        ).to(device)


        state = torch.load(
            ckpt,
            map_location=device
        )


        m.load_state_dict(
            state
        )


        m.eval()


        for p in m.parameters():

            p.requires_grad = False


        models.append(m)


    return models


# ============================================================
# LOAD T1
# ============================================================

print(
    f"\nLoading T1 checkpoints from:\n"
    f"{T1_CKPT_DIR}"
)


t1_models = load_checkpoints(

    MultiScaleSleepNetPlain,

    T1_CKPT_DIR,

    SEEDS,

    in_ch=3
)


print(
    f"  Loaded {len(t1_models)}/5"
)


# ============================================================
# LOAD T2
# ============================================================

print(
    f"\nLoading T2 checkpoints from:\n"
    f"{T2_CKPT_DIR}"
)


t2_models = load_checkpoints(

    ContextCNNTransformerSupCon,

    T2_CKPT_DIR,

    SEEDS,

    in_ch=3
)


print(
    f"  Loaded {len(t2_models)}/5"
)


# ============================================================
# LOAD T3
# ============================================================

print(
    f"\nLoading T3 checkpoints from:\n"
    f"{T3_CKPT_DIR}"
)


t3_models = load_checkpoints(

    TCNBiLSTM,

    T3_CKPT_DIR,

    SEEDS,

    in_channels=3
)


print(
    f"  Loaded {len(t3_models)}/5"
)


# ============================================================
# CHECK
# ============================================================

for name, models in [

    ("T1", t1_models),

    ("T2", t2_models),

    ("T3", t3_models)

]:

    if len(models) == 0:

        raise RuntimeError(

            f"No checkpoints loaded for {name}."
        )


# ============================================================
# INFERENCE HELPERS
# ============================================================

@torch.no_grad()
def arch_ensemble_probs_t1_or_t3(
    models,
    x
):

    probs_sum = None


    for m in models:

        logits = m(x)


        probs = F.softmax(
            logits,
            dim=1
        )


        if probs_sum is None:

            probs_sum = probs

        else:

            probs_sum = (
                probs_sum
                +
                probs
            )


    return (
        probs_sum
        /
        len(models)
    )


@torch.no_grad()
def arch_ensemble_probs_t2(
    models,
    x
):

    probs_sum = None


    for m in models:

        logits, _ = m(x)


        probs = F.softmax(
            logits,
            dim=1
        )


        if probs_sum is None:

            probs_sum = probs

        else:

            probs_sum = (
                probs_sum
                +
                probs
            )


    return (
        probs_sum
        /
        len(models)
    )


# ============================================================
# METRICS
# ============================================================

def compute_metrics(
    labels,
    preds
):

    acc = accuracy_score(
        labels,
        preds
    )


    f1 = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )


    kappa = cohen_kappa_score(
        labels,
        preds
    )


    per_cls = f1_score(

        labels,

        preds,

        average=None,

        zero_division=0,

        labels=[
            0,
            1,
            2,
            3,
            4
        ]
    )


    return (
        acc,
        f1,
        kappa,
        per_cls
    )


# ============================================================
# EVALUATE
# ============================================================

@torch.no_grad()
def evaluate_all(loader):

    all_labels = []

    t1_preds = []

    t2_preds = []

    t3_preds = []

    combined_preds = []


    for batch_idx, (x, y) in enumerate(loader):

        x = x.to(device)


        all_labels.extend(
            y.numpy()
        )


        # --------------------------------
        # T1
        # --------------------------------

        p1 = arch_ensemble_probs_t1_or_t3(

            t1_models,
            x
        )


        # --------------------------------
        # T2
        # --------------------------------

        p2 = arch_ensemble_probs_t2(

            t2_models,
            x
        )


        # --------------------------------
        # T3
        # --------------------------------

        p3 = arch_ensemble_probs_t1_or_t3(

            t3_models,
            x
        )


        # --------------------------------
        # Level-2 ensemble
        # Equal 1/3 weighting
        # --------------------------------

        combined = (
            p1
            +
            p2
            +
            p3
        ) / 3.0


        t1_preds.extend(

            p1.argmax(
                dim=1
            ).cpu().numpy()
        )


        t2_preds.extend(

            p2.argmax(
                dim=1
            ).cpu().numpy()
        )


        t3_preds.extend(

            p3.argmax(
                dim=1
            ).cpu().numpy()
        )


        combined_preds.extend(

            combined.argmax(
                dim=1
            ).cpu().numpy()
        )


        if (
            batch_idx + 1
        ) % 50 == 0:

            print(
                f"  Processed "
                f"{batch_idx + 1} batches..."
            )


    all_labels = np.array(
        all_labels
    )


    results = {}


    model_predictions = [

        (
            "T1: MultiScaleSleepNetPlain (5-seed ens)",
            t1_preds
        ),

        (
            "T2: Context-CNN-Transformer-SupCon (5-seed ens)",
            t2_preds
        ),

        (
            "T3: TCN-BiLSTM (5-seed ens)",
            t3_preds
        ),

        (
            "COMBINED (3-architecture ensemble)",
            combined_preds
        )
    ]


    for name, preds in model_predictions:

        preds = np.array(
            preds
        )


        acc, f1, kappa, per_cls = compute_metrics(

            all_labels,
            preds
        )


        results[name] = {

            "acc": acc,

            "f1": f1,

            "kappa": kappa,

            "per_cls": per_cls
        }


    return (
        results,
        all_labels,
        np.array(combined_preds)
    )


# ============================================================
# RUN VAL
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "Evaluating on VAL set"
)

print(
    "=" * 70
)


val_results, val_labels, val_combined_preds = evaluate_all(

    val_loader
)


# ============================================================
# RUN TEST
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "Evaluating on TEST set"
)

print(
    "=" * 70
)


test_results, test_labels, test_combined_preds = evaluate_all(

    test_loader
)


# ============================================================
# PRINT TABLE
# ============================================================

def print_table(
    results,
    split_name
):

    print(
        "\n"
        + "=" * 80
    )


    print(
        f"{split_name} SET RESULTS"
    )


    print(
        "=" * 80
    )


    print(
        f"{'Model':<55}"
        f"{'Acc%':>8}"
        f"{'F1':>8}"
        f"{'Kappa':>8}"
    )


    print(
        "-" * 80
    )


    for name, r in results.items():

        print(

            f"{name:<55}"

            f"{r['acc'] * 100:>8.2f}"

            f"{r['f1']:>8.4f}"

            f"{r['kappa']:>8.4f}"
        )


    print(
        "-" * 80
    )


    combined_key = [

        k

        for k in results

        if k.startswith("COMBINED")

    ][0]


    combined_per_cls = results[
        combined_key
    ]["per_cls"]


    print(
        f"\nCombined ensemble "
        f"per-class F1 ({split_name}):"
    )


    for i, name in enumerate(
        LABEL_NAMES
    ):

        print(

            f"  {name:6s}: "
            f"{combined_per_cls[i]:.4f}"
        )


# ============================================================
# PRINT VAL + TEST
# ============================================================

print_table(
    val_results,
    "VAL"
)


print_table(
    test_results,
    "TEST"
)


# ============================================================
# DILUTION CHECK
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "DILUTION CHECK (TEST set)"
)

print(
    "=" * 70
)


t1_key = [

    k

    for k in test_results

    if k.startswith("T1")

][0]


combined_key = [

    k

    for k in test_results

    if k.startswith("COMBINED")

][0]


t1_acc = (
    test_results[t1_key]["acc"]
    * 100
)


combined_acc = (
    test_results[combined_key]["acc"]
    * 100
)


t1_f1 = test_results[
    t1_key
]["f1"]


combined_f1 = test_results[
    combined_key
]["f1"]


print(
    f"Best single architecture (T1) : "
    f"Acc={t1_acc:.2f}%  F1={t1_f1:.4f}"
)


print(
    f"Combined 3-architecture ens.  : "
    f"Acc={combined_acc:.2f}%  F1={combined_f1:.4f}"
)


diff_acc = (
    combined_acc
    -
    t1_acc
)


diff_f1 = (
    combined_f1
    -
    t1_f1
)


print(
    f"\nDifference: "
    f"{diff_acc:+.2f}pp accuracy, "
    f"{diff_f1:+.4f} F1"
)


if diff_f1 > 0:

    print(
        "-> Combined ensemble BEATS "
        "the best single architecture."
    )

    print(
        "   Multi-teacher KD with this "
        "combination is a reasonable next step."
    )

else:

    print(
        "-> Combined ensemble does NOT "
        "beat T1 alone."
    )

    print(
        "   T2/T3 may be diluting T1."
    )

    print(
        "   If you use weighted ensemble, "
        "choose weights using VAL only."
    )


# ============================================================
# SAVE 1: COMBINED PREDICTIONS
# ============================================================

val_pred_path = os.path.join(
    OUTPUT_DIR,
    "val_combined_preds.npy"
)


test_pred_path = os.path.join(
    OUTPUT_DIR,
    "test_combined_preds.npy"
)


np.save(
    val_pred_path,
    val_combined_preds
)


np.save(
    test_pred_path,
    test_combined_preds
)


# ============================================================
# SAVE 2: GROUND-TRUTH LABELS
# ============================================================

val_label_path = os.path.join(
    OUTPUT_DIR,
    "val_labels.npy"
)


test_label_path = os.path.join(
    OUTPUT_DIR,
    "test_labels.npy"
)


np.save(
    val_label_path,
    val_labels
)


np.save(
    test_label_path,
    test_labels
)


# ============================================================
# SAVE 3: CSV RESULTS
# ============================================================

rows = []


for split_name, results in [

    (
        "VAL",
        val_results
    ),

    (
        "TEST",
        test_results
    )

]:

    for model_name, r in results.items():

        row = {

            "Split": split_name,

            "Model": model_name,

            "Accuracy": r["acc"],

            "Accuracy_percent":
                r["acc"] * 100,

            "Macro_F1": r["f1"],

            "Kappa": r["kappa"]
        }


        for i, cls_name in enumerate(
            LABEL_NAMES
        ):

            row[
                f"F1_{cls_name}"
            ] = r["per_cls"][i]


        rows.append(row)


results_df = pd.DataFrame(
    rows
)


csv_path = os.path.join(
    OUTPUT_DIR,
    "ensemble_results.csv"
)


results_df.to_csv(
    csv_path,
    index=False
)


# ============================================================
# SAVE 4: HUMAN-READABLE TXT REPORT
# ============================================================

txt_path = os.path.join(
    OUTPUT_DIR,
    "ensemble_results.txt"
)


with open(
    txt_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "=" * 80
        + "\n"
    )


    f.write(
        "MULTI-TEACHER ENSEMBLE "
        "EVALUATION RESULTS\n"
    )


    f.write(
        "=" * 80
        + "\n\n"
    )


    f.write(
        "Architectures:\n"
    )


    f.write(
        "T1 = MultiScaleSleepNetPlain\n"
    )


    f.write(
        "T2 = Context-CNN-Transformer-SupCon\n"
    )


    f.write(
        "T3 = TCN-BiLSTM\n\n"
    )


    f.write(
        "Ensemble strategy:\n"
    )


    f.write(
        "Level 1: 5-seed soft-vote "
        "per architecture\n"
    )


    f.write(
        "Level 2: Equal 1/3 weighting "
        "across architectures\n\n"
    )


    # --------------------------------------------------------
    # VAL + TEST
    # --------------------------------------------------------

    for split_name, results in [

        (
            "VAL",
            val_results
        ),

        (
            "TEST",
            test_results
        )

    ]:

        f.write(
            "\n"
            + "=" * 80
            + "\n"
        )


        f.write(
            f"{split_name} SET RESULTS\n"
        )


        f.write(
            "=" * 80
            + "\n\n"
        )


        for model_name, r in results.items():

            f.write(
                f"{model_name}\n"
            )


            f.write(
                f"  Accuracy : "
                f"{r['acc'] * 100:.2f}%\n"
            )


            f.write(
                f"  Macro F1 : "
                f"{r['f1']:.4f}\n"
            )


            f.write(
                f"  Kappa    : "
                f"{r['kappa']:.4f}\n"
            )


            f.write(
                "  Per-class F1:\n"
            )


            for i, cls_name in enumerate(
                LABEL_NAMES
            ):

                f.write(

                    f"    {cls_name:6s}: "
                    f"{r['per_cls'][i]:.4f}\n"
                )


            f.write(
                "\n"
            )


    # --------------------------------------------------------
    # DILUTION CHECK
    # --------------------------------------------------------

    f.write(
        "\n"
        + "=" * 80
        + "\n"
    )


    f.write(
        "DILUTION CHECK (TEST)\n"
    )


    f.write(
        "=" * 80
        + "\n\n"
    )


    f.write(
        f"T1 Accuracy       : "
        f"{t1_acc:.2f}%\n"
    )


    f.write(
        f"Combined Accuracy : "
        f"{combined_acc:.2f}%\n"
    )


    f.write(
        f"Accuracy Difference: "
        f"{diff_acc:+.2f} pp\n\n"
    )


    f.write(
        f"T1 Macro F1       : "
        f"{t1_f1:.4f}\n"
    )


    f.write(
        f"Combined Macro F1 : "
        f"{combined_f1:.4f}\n"
    )


    f.write(
        f"F1 Difference     : "
        f"{diff_f1:+.4f}\n"
    )


# ============================================================
# FINAL SAVE CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "RESULTS SAVED SUCCESSFULLY"
)

print(
    "=" * 80
)


print(
    f"\nOutput directory:\n"
    f"{OUTPUT_DIR}"
)


print(
    "\nSaved files:"
)


print(
    "  1. ensemble_results.csv"
)


print(
    "  2. ensemble_results.txt"
)


print(
    "  3. val_combined_preds.npy"
)


print(
    "  4. test_combined_preds.npy"
)


print(
    "  5. val_labels.npy"
)


print(
    "  6. test_labels.npy"
)


print(
    "\nDone!"
)

Device: cuda
Multi-teacher ensemble: T1 (MultiScaleSleepNetPlain) + T2 (Context-CNN-Transformer-SupCon) + T3 (TCN-BiLSTM)
Output directory: D:\22\AA\AA journal\evaluation\multi_teacher_ensemble
Val subjects: 11   Test subjects: 20

Building VAL dataset:
  Subjects: 11   Samples: 10,742
Building TEST dataset:
  Subjects: 20   Samples: 19,763

Loading T1 checkpoints from:
D:\22\AA\AA journal\evaluation\multiscale_plain_c7_valfixed
  Loading seed 42...
  Loading seed 123...
  Loading seed 256...
  Loading seed 789...
  Loading seed 999...
  Loaded 5/5

Loading T2 checkpoints from:
D:\22\AA\AA journal\evaluation\context_cnn_transformer_c7_supcon_valfixed
  Loading seed 42...


c:\Users\pc\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  Loading seed 123...
  Loading seed 256...
  Loading seed 789...
  Loading seed 999...
  Loaded 5/5

Loading T3 checkpoints from:
D:\22\AA\AA journal\evaluation\tcn_bilstm_c7_valfixed
  Loading seed 42...
  Loading seed 123...
  Loading seed 256...
  Loading seed 789...
  Loading seed 999...
  Loaded 5/5

Evaluating on VAL set
  Processed 50 batches...
  Processed 100 batches...
  Processed 150 batches...

Evaluating on TEST set
  Processed 50 batches...
  Processed 100 batches...
  Processed 150 batches...
  Processed 200 batches...
  Processed 250 batches...
  Processed 300 batches...

VAL SET RESULTS
Model                                                      Acc%      F1   Kappa
--------------------------------------------------------------------------------
T1: MultiScaleSleepNetPlain (5-seed ens)                  85.28  0.8136  0.8053
T2: Context-CNN-Transformer-SupCon (5-seed ens)           85.14  0.8126  0.8030
T3: TCN-BiLSTM (5-seed ens)                               85.29  0.